# SR 11-7 Model Validation Toolkit — Demo & Usage Guide

**Propósito:** Este notebook demuestra el uso estándar de `sr117_validator` para el equipo de validación. Cada sección corresponde a un tipo de prueba definido en la guía SR 11-7 de la Reserva Federal.

**Secciones:**
1. Instalación y configuración
2. `compare_dataframes` — Benchmarking / Dry Run vs Model Owner
3. `vif_check` — Test de multicolinealidad (supuestos de regresión)
4. `psi_check` — Population Stability Index (estabilidad de inputs)
5. `csi_check` — Characteristic Stability Index (estabilidad por segmento)
6. Exportar evidencia a Excel (trazabilidad para auditoría)
7. Flujo completo de validación

---
> **Nota:** Reemplaza los DataFrames sintéticos por los datos reales del modelo bajo revisión. La estructura de inputs y la interpretación de outputs es idéntica.

## 1. Instalación y configuración

Instalar desde PyPI (una sola vez por ambiente):

In [ ]:
# Instalar la librería
# !pip install sr117-validator

# Para desarrollo local (desde el repo clonado):
# !pip install -e path/to/sr117_validator

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime
import os

# Importar funciones de la librería
from validation_suite import (
    compare_dataframes,
    vif_check,
    psi_check,
    csi_check,
    ValidationResult,
)

pd.set_option('display.float_format', '{:.6f}'.format)
pd.set_option('display.max_columns', 20)

print("Librería cargada correctamente.")

---
## 2. `compare_dataframes` — Dry Run vs Model Owner

### Contexto SR 11-7
La sección **IV.A** del SR 11-7 requiere que el validador replique de forma independiente los outputs del model owner. `compare_dataframes` estandariza esa comparación: verifica schema, alinea por llave primaria, y cuantifica diferencias numéricas con tolerancias configurables.

**Cuándo usarla:**
- Al comparar tu corrida independiente (dry run) contra los outputs oficiales del dueño del modelo
- Al verificar que una nueva versión del modelo produce los mismos outputs que la versión en producción
- Al comparar outputs de dos entornos distintos (dev vs prod)

### 2.1 Datos de ejemplo — Modelo PD (Probability of Default)

In [ ]:
# ── REEMPLAZAR CON DATOS REALES ─────────────────────────────────────────────
# df_model_owner  = pd.read_csv('path/to/model_owner_output.csv')
# df_dry_run      = pd.read_csv('path/to/validator_dryrun.csv')
# ────────────────────────────────────────────────────────────────────────────

rng = np.random.default_rng(42)
n = 500

obligor_ids = [f"OBL-{i:05d}" for i in range(1, n + 1)]
pd_scores   = rng.beta(a=2, b=18, size=n)
lgd_values  = rng.beta(a=4, b=6,  size=n)
ead_values  = rng.lognormal(mean=11.5, sigma=1.2, size=n)

df_model_owner = pd.DataFrame({
    'obligor_id': obligor_ids,
    'pd_score':   pd_scores,
    'lgd':        lgd_values,
    'ead':        ead_values,
    'segment':    rng.choice(['Retail', 'SME', 'Corporate'], size=n),
})

# Dry run con 8 diferencias numéricas pequeñas
df_dry_run = df_model_owner.copy()
idx_diff = rng.choice(n, size=8, replace=False)
df_dry_run.loc[idx_diff, 'pd_score'] += rng.uniform(0.001, 0.008, size=8)

print(f"Model Owner rows : {len(df_model_owner):,}")
print(f"Dry Run rows     : {len(df_dry_run):,}")
print(f"Rows with delta  : {len(idx_diff)}")
df_model_owner.head()

### 2.2 Caso 1 — Comparación con tolerancia estricta (default)

In [ ]:
result_strict = compare_dataframes(
    df_reference=df_model_owner,
    df_challenger=df_dry_run,
    key_cols=['obligor_id'],
    numeric_tol=1e-6,
    label_reference='Model Owner v2.3',
    label_challenger='Validator Dry Run',
)

print(f"Status   : {result_strict.status}")
print(f"Warnings : {result_strict.warnings if result_strict.warnings else 'None'}")
print()
result_strict.summary_df.sort_values('max_abs_diff', ascending=False)

**Interpretación:** Las columnas con `rows_exceeding_tol > 0` requieren análisis adicional. Una diferencia numérica pequeña (ej. `max_abs_diff < 0.005`) puede ser aceptable si se explica por diferencias de redondeo entre plataformas — documentar en el reporte de validación.

### 2.3 Caso 2 — Tolerancia permisiva (diferencias de redondeo aceptables)

In [ ]:
result_permissive = compare_dataframes(
    df_reference=df_model_owner,
    df_challenger=df_dry_run,
    key_cols=['obligor_id'],
    numeric_tol=0.01,
    label_reference='Model Owner v2.3',
    label_challenger='Validator Dry Run',
)

print(f"Status con tol=1e-6 : {result_strict.status}")
print(f"Status con tol=0.01 : {result_permissive.status}")
print()
result_permissive.summary_df[['column', 'max_abs_diff', 'rows_exceeding_tol']]

### 2.4 Caso 3 — Schema mismatch (columnas faltantes)

In [ ]:
df_incomplete = df_dry_run.drop(columns=['ead'])

result_schema = compare_dataframes(
    df_reference=df_model_owner,
    df_challenger=df_incomplete,
    key_cols=['obligor_id'],
)

print("Schema warnings detectados:")
for w in result_schema.warnings:
    print(f"  ⚠  {w}")

print(f"\nStatus: {result_schema.status}")
print("\nColumnas que pudieron compararse:")
result_schema.summary_df[['column', 'max_abs_diff', 'rows_exceeding_tol']]

---
## 3. `vif_check` — Test de Multicolinealidad

### Contexto SR 11-7
El SR 11-7 exige validar los **supuestos del modelo**. Para regresiones (OLS, logística), la multicolinealidad entre predictores infla los errores estándar y hace los coeficientes inestables.

**Umbrales de referencia SR 11-7:**

| VIF | Clasificación | Acción recomendada |
|-----|--------------|-------------------|
| < 5 | OK | Sin acción |
| 5 – 10 | MODERATE | Monitorear, documentar justificación |
| > 10 | HIGH | Requiere remediación o justificación fuerte |

In [ ]:
# ── REEMPLAZAR CON DATOS REALES ─────────────────────────────────────────────
# df_features = pd.read_csv('path/to/model_features.csv')
# feature_cols = ['ltv', 'dti', 'credit_age', 'utilization_rate']
# ────────────────────────────────────────────────────────────────────────────

rng2 = np.random.default_rng(7)
n_obs = 1000

df_features_clean = pd.DataFrame({
    'ltv':              rng2.uniform(0.30, 0.95, n_obs),
    'dti':              rng2.uniform(0.10, 0.60, n_obs),
    'credit_age_yrs':   rng2.uniform(1,    30,   n_obs),
    'utilization_rate': rng2.uniform(0,    1,    n_obs),
    'num_delinquencies':rng2.poisson(lam=0.4, size=n_obs).astype(float),
})

# Feature redundante — dti_annualized es dti * 12
df_features_collinear = df_features_clean.copy()
df_features_collinear['dti_annualized'] = (
    df_features_collinear['dti'] * 12 + rng2.normal(0, 0.001, n_obs)
)

FEATURE_COLS     = ['ltv', 'dti', 'credit_age_yrs', 'utilization_rate', 'num_delinquencies']
FEATURES_COLLINEAR = FEATURE_COLS + ['dti_annualized']

print("DataFrames de features listos.")

In [ ]:
result_vif_clean = vif_check(df_features_clean, feature_cols=FEATURE_COLS, threshold=10.0)
result_vif_collinear = vif_check(df_features_collinear, feature_cols=FEATURES_COLLINEAR, threshold=10.0)

print(f"VIF sin colinealidad  → Status: {result_vif_clean.status}")
print(f"VIF con feature extra → Status: {result_vif_collinear.status}")
print()
result_vif_collinear.summary_df.sort_values('VIF', ascending=False)

In [ ]:
def plot_vif(result: ValidationResult, title: str = "VIF por variable") -> None:
    """Bar chart de VIF con bandas de umbral SR 11-7."""
    df = result.summary_df.sort_values('VIF', ascending=True)
    color_map = {'OK': '#1D9E75', 'MODERATE': '#BA7517', 'HIGH': '#E24B4A'}
    colors = df['flag'].map(color_map)

    fig, ax = plt.subplots(figsize=(8, max(3, len(df) * 0.5)))
    bars = ax.barh(df['feature'], df['VIF'], color=colors, height=0.55, alpha=0.85)
    ax.axvline(x=5,  color='#BA7517', linestyle='--', linewidth=1, alpha=0.7)
    ax.axvline(x=10, color='#E24B4A', linestyle='--', linewidth=1, alpha=0.7)
    for bar, val in zip(bars, df['VIF']):
        ax.text(val + 0.1, bar.get_y() + bar.get_height() / 2,
                f'{val:.2f}', va='center', fontsize=9)
    patches = [
        mpatches.Patch(color='#1D9E75', label='OK (VIF < 5)'),
        mpatches.Patch(color='#BA7517', label='Moderate (5–10)'),
        mpatches.Patch(color='#E24B4A', label='High (> 10)'),
    ]
    ax.legend(handles=patches, loc='lower right', fontsize=8)
    status_color = '#1D9E75' if result.status == 'PASS' else '#E24B4A'
    ax.set_title(f"{title}  |  Status: {result.status}",
                 fontsize=11, color=status_color, fontweight='bold')
    ax.set_xlabel('Variance Inflation Factor (VIF)', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()


plot_vif(result_vif_clean,     title="VIF — Modelo base")
plot_vif(result_vif_collinear, title="VIF — Modelo con feature redundante")

---
## 4. `psi_check` — Population Stability Index

### Contexto SR 11-7
El PSI mide si la **distribución de las variables de input** ha cambiado entre el período de desarrollo del modelo y el período de monitoreo actual. Un PSI elevado indica que el modelo está siendo aplicado a una población diferente de aquella para la que fue construido — trigger directo de recalibración según SR 11-7.

**Umbrales estándar SR 11-7:**

| PSI | Clasificación | Acción recomendada |
|-----|--------------|-------------------|
| < 0.10 | STABLE | Sin acción |
| 0.10 – 0.25 | MODERATE | Investigar causa, monitorear |
| > 0.25 | UNSTABLE | Recalibración o rediseño del modelo |

**Cuándo usarla:**
- En cada ciclo de monitoreo periódico (trimestral/anual)
- Al comparar el portafolio de desarrollo contra el portafolio vigente
- Antes de aplicar un modelo a un nuevo segmento o mercado

### 4.1 Datos de ejemplo — portafolio de desarrollo vs monitoreo

In [ ]:
# ── REEMPLAZAR CON DATOS REALES ─────────────────────────────────────────────
# df_dev      = pd.read_csv('path/to/development_sample.csv')
# df_monitor  = pd.read_csv('path/to/monitoring_sample.csv')
# psi_columns = ['pd_score', 'ltv', 'dti']
# ────────────────────────────────────────────────────────────────────────────

rng3 = np.random.default_rng(20)
n_dev = 5000
n_mon = 5000

# Muestra de desarrollo — distribuciones de referencia
df_dev = pd.DataFrame({
    'pd_score': rng3.beta(2, 18, n_dev),          # PD típico portafolio retail
    'ltv':      rng3.uniform(0.30, 0.90, n_dev),   # Estable entre períodos
    'dti':      rng3.uniform(0.10, 0.55, n_dev),   # Estable entre períodos
})

# Muestra de monitoreo — simula deterioro de calidad crediticia
# pd_score se desplaza hacia la derecha (más riesgo)
# ltv y dti se mantienen estables
df_monitor = pd.DataFrame({
    'pd_score': rng3.beta(5, 10, n_mon),           # shift severo — portafolio deteriorado
    'ltv':      rng3.uniform(0.30, 0.90, n_mon),   # sin cambio
    'dti':      rng3.uniform(0.12, 0.57, n_mon),   # cambio mínimo
})

print(f"Desarrollo  : {n_dev:,} observaciones  |  Período: 2021-Q1 (referencia)")
print(f"Monitoreo   : {n_mon:,} observaciones  |  Período: 2025-Q4 (actual)")
print()
print("Estadísticos descriptivos — pd_score:")
pd.DataFrame({
    'Desarrollo': df_dev['pd_score'].describe(),
    'Monitoreo':  df_monitor['pd_score'].describe(),
}).round(4)

### 4.2 PSI para todas las variables de input

In [ ]:
PSI_COLUMNS = ['pd_score', 'ltv', 'dti']

result_psi = psi_check(
    expected=df_dev,
    actual=df_monitor,
    columns=PSI_COLUMNS,
    bins=10,           # deciles — convención SR 11-7
)

print(f"Status general: {result_psi.status}")
print()
if result_psi.warnings:
    print("Warnings:")
    for w in result_psi.warnings:
        print(f"  ⚠  {w}")
    print()

result_psi.summary_df.sort_values('psi_value', ascending=False)

### 4.3 Visualización — PSI por variable

In [ ]:
def plot_psi_summary(result: ValidationResult, title: str = "PSI por variable") -> None:
    """Bar chart de PSI con bandas de umbral SR 11-7."""
    df = result.summary_df.sort_values('psi_value', ascending=True)
    color_map = {'STABLE': '#1D9E75', 'MODERATE': '#BA7517', 'UNSTABLE': '#E24B4A'}
    colors = df['flag'].map(color_map)

    fig, ax = plt.subplots(figsize=(8, max(3, len(df) * 0.6)))
    bars = ax.barh(df['variable'], df['psi_value'], color=colors, height=0.5, alpha=0.85)

    ax.axvline(x=0.10, color='#BA7517', linestyle='--', linewidth=1.2, alpha=0.8, label='Moderate (0.10)')
    ax.axvline(x=0.25, color='#E24B4A', linestyle='--', linewidth=1.2, alpha=0.8, label='Unstable (0.25)')

    for bar, val in zip(bars, df['psi_value']):
        ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', fontsize=9)

    patches = [
        mpatches.Patch(color='#1D9E75', label='Stable (< 0.10)'),
        mpatches.Patch(color='#BA7517', label='Moderate (0.10–0.25)'),
        mpatches.Patch(color='#E24B4A', label='Unstable (> 0.25)'),
    ]
    ax.legend(handles=patches, loc='lower right', fontsize=8)
    status_color = '#1D9E75' if result.status == 'PASS' else '#E24B4A'
    ax.set_title(f"{title}  |  Status: {result.status}",
                 fontsize=11, color=status_color, fontweight='bold')
    ax.set_xlabel('PSI', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()


plot_psi_summary(result_psi, title="PSI — Desarrollo 2021-Q1 vs Monitoreo 2025-Q4")

### 4.4 Detalle por bin — diagnóstico de dónde ocurre el shift

In [ ]:
def plot_psi_bins(result: ValidationResult, variable: str) -> None:
    """Visualiza la distribución esperada vs actual por bin para una variable."""
    detail = result.details['bin_detail'][variable].copy()
    psi_total = detail['psi_contribution'].sum()
    flag = result.summary_df.loc[
        result.summary_df['variable'] == variable, 'flag'
    ].iloc[0]

    x = range(len(detail))
    width = 0.35

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

    # Panel izquierdo: distribuciones esperada vs actual
    ax1.bar([i - width/2 for i in x], detail['expected_pct'],
            width, label='Desarrollo (expected)', color='#378ADD', alpha=0.75)
    ax1.bar([i + width/2 for i in x], detail['actual_pct'],
            width, label='Monitoreo (actual)', color='#E24B4A', alpha=0.75)
    ax1.set_xticks(list(x))
    ax1.set_xticklabels(
        [f'B{i+1}' for i in x], rotation=45, fontsize=7
    )
    ax1.set_ylabel('Proporción')
    ax1.set_title(f'{variable} — Distribución por decil', fontsize=10)
    ax1.legend(fontsize=8)
    ax1.spines[['top', 'right']].set_visible(False)

    # Panel derecho: contribución PSI por bin
    bar_colors = [
        '#E24B4A' if v > 0.025 else '#BA7517' if v > 0.010 else '#1D9E75'
        for v in detail['psi_contribution']
    ]
    ax2.bar(list(x), detail['psi_contribution'], color=bar_colors, alpha=0.85)
    ax2.axhline(y=0.025, color='#E24B4A', linestyle='--', linewidth=0.8, alpha=0.7)
    ax2.set_xticks(list(x))
    ax2.set_xticklabels(
        [f'B{i+1}' for i in x], rotation=45, fontsize=7
    )
    ax2.set_ylabel('Contribución PSI')
    flag_color = '#1D9E75' if flag == 'STABLE' else '#BA7517' if flag == 'MODERATE' else '#E24B4A'
    ax2.set_title(
        f'{variable} — PSI = {psi_total:.4f}  [{flag}]',
        fontsize=10, color=flag_color, fontweight='bold'
    )
    ax2.spines[['top', 'right']].set_visible(False)

    plt.tight_layout()
    plt.show()


# Examinar la variable más crítica
plot_psi_bins(result_psi, 'pd_score')
plot_psi_bins(result_psi, 'ltv')

**Interpretación del detalle por bin:** Los bins con contribución PSI > 0.025 (línea roja punteada) son los deciles donde la distribución cambió más. En el panel izquierdo se puede ver hacia qué dirección se desplazó la población — hacia la derecha indica mayor riesgo.

### 4.5 PSI aceptando Series directamente

In [ ]:
# También funciona con pd.Series — útil cuando trabajas con una sola variable
result_psi_single = psi_check(
    expected=df_dev['pd_score'],
    actual=df_monitor['pd_score'],
    bins=10,
)

row = result_psi_single.summary_df.iloc[0]
print(f"Variable         : {row['variable']}")
print(f"PSI              : {row['psi_value']:.4f}")
print(f"Flag             : {row['flag']}")
print(f"Bins con shift   : {row['n_bins_shifted']} de {result_psi_single.details['bins']}")

---
## 5. `csi_check` — Characteristic Stability Index

### Contexto SR 11-7
El CSI extiende el PSI al nivel de **segmento**: mide el shift distribucional del score del modelo dentro de cada categoría de una variable de segmentación (risk grade, producto, geografía). Donde el PSI da una señal global, el CSI pinpoints exactamente qué segmento está generando la inestabilidad.

**Cuándo usarla:**
- Después de un PSI elevado, para identificar el segmento driver
- En modelos PD donde los risk grades son el segmento natural
- En modelos aplicados a múltiples líneas de producto o geografías
- Como test de estabilidad post-merger o post-acquisición

### 5.1 Datos de ejemplo — score PD por risk grade

In [ ]:
# ── REEMPLAZAR CON DATOS REALES ─────────────────────────────────────────────
# df_dev_seg     = pd.read_csv('path/to/development_segmented.csv')
# df_monitor_seg = pd.read_csv('path/to/monitoring_segmented.csv')
# SCORE_COL      = 'pd_score'
# SEGMENT_COL    = 'risk_grade'
# ────────────────────────────────────────────────────────────────────────────

rng4 = np.random.default_rng(3)
n_per_seg = 800

SCORE_COL   = 'pd_score'
SEGMENT_COL = 'risk_grade'

def _make_segment(rng, segment, a, b, n):
    return pd.DataFrame({
        'pd_score':   rng.beta(a, b, n),
        'risk_grade': segment,
    })

# Desarrollo: distribuciones de referencia por grade
df_dev_seg = pd.concat([
    _make_segment(rng4, 'AAA', a=1,  b=40, n=n_per_seg),   # muy bajo riesgo
    _make_segment(rng4, 'BBB', a=3,  b=15, n=n_per_seg),   # riesgo medio
    _make_segment(rng4, 'CCC', a=8,  b=5,  n=n_per_seg),   # riesgo alto
], ignore_index=True)

# Monitoreo: AAA y BBB estables, CCC con deterioro severo
df_monitor_seg = pd.concat([
    _make_segment(rng4, 'AAA', a=1,   b=40, n=n_per_seg),  # estable
    _make_segment(rng4, 'BBB', a=4,   b=12, n=n_per_seg),  # shift moderado
    _make_segment(rng4, 'CCC', a=15,  b=3,  n=n_per_seg),  # deterioro severo
], ignore_index=True)

print("Estadísticos de pd_score por risk grade:")
print("\n--- Desarrollo ---")
print(df_dev_seg.groupby('risk_grade')['pd_score'].agg(['mean','std','count']).round(4))
print("\n--- Monitoreo ---")
print(df_monitor_seg.groupby('risk_grade')['pd_score'].agg(['mean','std','count']).round(4))

### 5.2 CSI por risk grade

In [ ]:
result_csi = csi_check(
    expected=df_dev_seg,
    actual=df_monitor_seg,
    score_col=SCORE_COL,
    segment_col=SEGMENT_COL,
    bins=10,
)

print(f"Status general: {result_csi.status}")
print()
if result_csi.warnings:
    print("Warnings:")
    for w in result_csi.warnings:
        print(f"  ⚠  {w}")
    print()

result_csi.summary_df.sort_values('csi_value', ascending=False)

### 5.3 Visualización — CSI por segmento con composición del portafolio

In [ ]:
def plot_csi(result: ValidationResult, title: str = "CSI por segmento") -> None:
    """Panel doble: CSI por segmento + composición del portafolio (expected vs actual)."""
    df = result.summary_df.sort_values('csi_value', ascending=False).reset_index(drop=True)
    color_map = {'STABLE': '#1D9E75', 'MODERATE': '#BA7517', 'UNSTABLE': '#E24B4A'}

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, max(4, len(df) * 0.7)))

    # Panel izquierdo: CSI por segmento
    colors = df['flag'].map(color_map)
    bars = ax1.barh(df['segment'].astype(str), df['csi_value'],
                    color=colors, height=0.5, alpha=0.85)
    ax1.axvline(x=0.10, color='#BA7517', linestyle='--', linewidth=1.2, alpha=0.8)
    ax1.axvline(x=0.25, color='#E24B4A', linestyle='--', linewidth=1.2, alpha=0.8)
    for bar, val in zip(bars, df['csi_value']):
        ax1.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
                 f'{val:.4f}', va='center', fontsize=9)
    patches = [
        mpatches.Patch(color='#1D9E75', label='Stable'),
        mpatches.Patch(color='#BA7517', label='Moderate'),
        mpatches.Patch(color='#E24B4A', label='Unstable'),
    ]
    ax1.legend(handles=patches, fontsize=8)
    status_color = '#1D9E75' if result.status == 'PASS' else '#E24B4A'
    ax1.set_title(f"CSI por segmento  |  Status: {result.status}",
                  fontsize=10, color=status_color, fontweight='bold')
    ax1.set_xlabel('CSI', fontsize=9)
    ax1.spines[['top', 'right']].set_visible(False)

    # Panel derecho: composición del portafolio
    x = range(len(df))
    width = 0.35
    ax2.bar([i - width/2 for i in x], df['segment_pct_exp'] * 100,
            width, label='Desarrollo (expected)', color='#378ADD', alpha=0.75)
    ax2.bar([i + width/2 for i in x], df['segment_pct_act'] * 100,
            width, label='Monitoreo (actual)', color='#E24B4A', alpha=0.75)
    ax2.set_xticks(list(x))
    ax2.set_xticklabels(df['segment'].astype(str), fontsize=9)
    ax2.set_ylabel('% del portafolio')
    ax2.set_title('Composición del portafolio por segmento', fontsize=10)
    ax2.legend(fontsize=8)
    ax2.spines[['top', 'right']].set_visible(False)

    plt.suptitle(title, fontsize=11, y=1.02)
    plt.tight_layout()
    plt.show()


plot_csi(result_csi, title="CSI — Desarrollo 2021-Q1 vs Monitoreo 2025-Q4")

**Interpretación del panel derecho:** Cambios en la composición del portafolio (% por segmento) son un hallazgo de MRM independiente del valor CSI. Si el segmento CCC pasó de 15% a 35% del portafolio, el modelo está siendo aplicado a un mix de riesgo radicalmente distinto al de desarrollo — documentar en el reporte como finding de concentración.

### 5.4 Detalle de bins para el segmento crítico

In [ ]:
# Identificar el segmento con mayor CSI
worst_segment = result_csi.summary_df.sort_values('csi_value', ascending=False).iloc[0]
print(f"Segmento más crítico : {worst_segment['segment']}")
print(f"CSI                  : {worst_segment['csi_value']:.4f}")
print(f"Flag                 : {worst_segment['flag']}")
print()

# Detalle por bin del segmento crítico
bin_detail = result_csi.details['bin_detail'][str(worst_segment['segment'])]
print("Detalle por bin:")
bin_detail.style.background_gradient(
    subset=['psi_contribution'], cmap='RdYlGn_r'
).format({'expected_pct': '{:.2%}', 'actual_pct': '{:.2%}',
          'psi_contribution': '{:.4f}'})

---
## 6. Exportar evidencia a Excel

Todo `ValidationResult` exporta a Excel con pestaña **Summary** + **Metadata** — listo para adjuntar al reporte de validación o al expediente de auditoría FED.

In [ ]:
os.makedirs('validation_evidence', exist_ok=True)
ts = datetime.now().strftime('%Y%m%d_%H%M')

export_map = {
    'compare_dryrun':   result_strict,
    'vif_check':        result_vif_collinear,
    'psi_check':        result_psi,
    'csi_check':        result_csi,
}

for name, result in export_map.items():
    path = f'validation_evidence/{name}_{ts}.xlsx'
    result.to_excel(path)
    print(f"  ✓  {path}")

---
## 7. Flujo completo de validación

Bloque integrado con todas las funciones disponibles. Modificar las constantes de configuración al inicio y correr de principio a fin para cada ejercicio de validación.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CONFIGURACIÓN — ADAPTAR PARA CADA VALIDACIÓN
# ═══════════════════════════════════════════════════════════════════

MODEL_ID        = 'PD-RETAIL-V2.3'
VALIDATOR       = 'Team Quant Validation'
REFERENCE_DT    = '2025-Q4'

KEY_COLS        = ['obligor_id']
FEATURE_COLS    = ['ltv', 'dti', 'credit_age_yrs', 'utilization_rate']
PSI_COLS        = ['pd_score', 'ltv', 'dti']
SCORE_COL       = 'pd_score'
SEGMENT_COL     = 'risk_grade'

VIF_THRESHOLD   = 10.0
NUMERIC_TOL     = 1e-4
PSI_BINS        = 10

# ── Cargar datos (reemplazar con paths reales) ──────────────────────
# df_mo       = pd.read_csv('model_owner_output.csv')
# df_val      = pd.read_csv('validator_dryrun.csv')
# df_feats    = pd.read_csv('model_features.csv')
# df_dev      = pd.read_csv('development_sample.csv')
# df_monitor  = pd.read_csv('monitoring_sample.csv')
# df_dev_seg  = pd.read_csv('development_segmented.csv')
# df_mon_seg  = pd.read_csv('monitoring_segmented.csv')

# Para este demo usamos los DataFrames ya construidos:
df_mo      = df_model_owner
df_val     = df_dry_run
df_feats   = df_features_collinear
df_dev_psi = df_dev
df_mon_psi = df_monitor
df_dev_csi = df_dev_seg
df_mon_csi = df_monitor_seg

print(f"Modelo          : {MODEL_ID}")
print(f"Período         : {REFERENCE_DT}")
print(f"Validador       : {VALIDATOR}")
print(f"Fecha ejecución : {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("─" * 55)

In [ ]:
# ── Ejecutar todos los tests ────────────────────────────────────────

r_compare = compare_dataframes(
    df_reference=df_mo, df_challenger=df_val,
    key_cols=KEY_COLS, numeric_tol=NUMERIC_TOL,
    label_reference=f'MO {MODEL_ID}', label_challenger='Validator Dry Run',
)

r_vif = vif_check(
    df=df_feats, feature_cols=FEATURE_COLS, threshold=VIF_THRESHOLD,
)

r_psi = psi_check(
    expected=df_dev_psi, actual=df_mon_psi,
    columns=PSI_COLS, bins=PSI_BINS,
)

r_csi = csi_check(
    expected=df_dev_csi, actual=df_mon_csi,
    score_col=SCORE_COL, segment_col=SEGMENT_COL, bins=PSI_BINS,
)

# ── Resumen ejecutivo ───────────────────────────────────────────────
all_results = {
    'Benchmarking (Dry Run)':       r_compare,
    'Supuestos: Multicolinealidad':  r_vif,
    'Stability: PSI':               r_psi,
    'Stability: CSI por segmento':  r_csi,
}

print(f"{'─'*55}")
print(f"RESUMEN EJECUTIVO — {MODEL_ID} | {REFERENCE_DT}")
print(f"{'─'*55}")
print(f"  {'Test':<35} {'Status':>6}  {'Warns':>5}")
print(f"{'─'*55}")

all_pass = True
for test_name, res in all_results.items():
    symbol = '✓' if res.status == 'PASS' else '✗'
    print(f"  {symbol} {test_name:<34} {res.status:>6}  {len(res.warnings):>5}")
    if res.status != 'PASS':
        all_pass = False

print(f"{'─'*55}")
overall = 'PASS' if all_pass else 'FAIL — revisar tests marcados con ✗'
print(f"  OVERALL: {overall}")
print()

# ── Exportar evidencia ──────────────────────────────────────────────
os.makedirs('validation_evidence', exist_ok=True)
ts = datetime.now().strftime('%Y%m%d_%H%M')
for test_name, res in all_results.items():
    slug = (
        test_name.lower()
        .replace(' ', '_')
        .replace('(', '').replace(')', '')
        .replace(':', '')
    )
    path = f'validation_evidence/{MODEL_ID}_{slug}_{ts}.xlsx'
    res.to_excel(path)
    print(f"  Evidencia → {path}")

---
## Referencia rápida

| Función | Cuándo usar | Inputs clave | Output clave |
|---|---|---|---|
| `compare_dataframes(ref, cha, key_cols)` | Dry run vs Model Owner | `numeric_tol`, `label_*` | `status`, `summary_df[max_abs_diff]` |
| `vif_check(df, feature_cols)` | Supuestos de regresión | `threshold` | `summary_df[VIF, flag]` |
| `psi_check(expected, actual)` | Estabilidad de inputs | `columns`, `bins` | `summary_df[psi_value, flag]`, `details[bin_detail]` |
| `csi_check(expected, actual, score_col, segment_col)` | Estabilidad por segmento | `bins` | `summary_df[csi_value, segment_pct_*]`, `details[bin_detail]` |
| `result.to_excel(path)` | Evidencia auditoría FED | — | `.xlsx` con Summary + Metadata |

**Próxima función:**
- `backtesting_report()` — Gini, AUC-ROC, KS statistic, Brier score, calibración PD

---
*Generado con `sr117_validator` — Alineado con SR 11-7 (Federal Reserve, 2011)*